# 7-dars: PyTorch — telefon qo’lyozma raqamni qanday taniydi?

**Hikoya.** Siz dugonangizning daftaridan uy ishi javobini rasmga oldingiz. Telefon rasmdagi **qo’lda yozilgan raqamlarni** o’zi o’qiy oladi. Bank ilovasi chekdagi summani, pochta esa konvertdagi indeksni shunday o’qiydi. Telefon qanday biladiki, bu **3**, bu esa **8**?

6-darsda siz kichik tarmoqni **noldan** yozdingiz: 17 ta son, 4 ta holat. Bugun xuddi shu g’oyani **PyTorch** da, **haqiqiy kutubxonada** yozamiz. Tarmoq **60 000 ta** qo’lyozma raqamdan o’rganadi.

**Bugun siz:**
- **tensor** nima ekanini ko’rasiz: sonlar jadvali, rasm ham shunday;
- 6-darsdagi `Value` ning **katta akasi** (PyTorch autograd) bitta qatorda ishlashini ko’rasiz;
- 6-darsdagi lampochka tarmog’ini PyTorch da **6 qatorda** qayta yasaysiz;
- raqam taniydigan tarmoqni o’qitib, **95% dan yuqori** aniqlikka erishasiz;
- tarmoq qaysi raqamlarda adashishini ko’rasiz va modelni **saqlaysiz**.

**Asosiy fikr:** PyTorch yangi g’oya emas. 6-darsdagi tsikl (**oldinga → loss → orqaga → qadam**) aynan o’sha. PyTorch faqat uni **qisqa, tez va katta ma’lumot bilan** qiladi.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**).

## Bugungi so’zlar (shu jadvalga qaytib turing)

| So’z | Oddiy tilda | Misol |
|---|---|---|
| **tensor** | sonlar «jadvali» (bitta son ham, ro’yxat ham, rasm ham) | rasm = 28 × 28 sondan iborat jadval |
| **shape** | tensor o’lchami: nechta qator, nechta ustun | `[2, 3]` — 2 qator, 3 ustun |
| **autograd** | ta’sirlarni (grad) kompyuter o’zi hisoblashi: 6-darsdagi `Value` | `L.backward()` |
| **`nn.Linear`** | yonma-yon turgan neyronlar qatlami | `nn.Linear(2, 4)`: 4 ta neyron |
| **model** | qatlamlar zanjiri (6-darsdagi «tarmoq») | `nn.Sequential(...)` |
| **optimizer** | og’irliklarni surib boradigan «haydovchi» (6-darsdagi `p.data - lr * p.grad`) | `optimizer.step()` |
| **batch** | birdaniga ko’riladigan rasmlar to’plami («daftarcha») | 64 ta rasm |
| **epoxa** | butun ma’lumotni **bir marta** ko’rib chiqish | 60 000 rasmning hammasi bir marta |
| **train / test** | o’rganish uchun / imtihon uchun ma’lumot | 60 000 / 10 000 |
| **aniqlik** | to’g’ri javoblar ulushi | 97% — 100 tadan 97 tasi to’g’ri |
| **GPU** | juda ko’p oddiy hisobni bir vaqtda qiladigan chip | o’yin kartasi |

## 0. Kutubxonalar

- `torch` — PyTorch, asosiy kutubxona;
- `torchvision` — tayyor rasm ma’lumotlari (bugun MNIST raqamlari);
- `matplotlib` — grafik va rasm chizish.

Colab’da PyTorch **tayyor o’rnatilgan**, qo’shimcha hech narsa o’rnatmaysiz.

In [ ]:
import time
import torch
from torch import nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print("PyTorch versiyasi:", torch.__version__)

## 1. Tensor — sonlar jadvali

6-darsda har son `Value` ichida edi. PyTorch’da sonlar **tensor** ichida turadi. Tensor — oddiy **sonlar jadvali**. Uch xil ko’rinishini ko’ramiz.

**Birinchi: bitta son.**

In [ ]:
son = torch.tensor(5.0)

print(son)
print("o'lcham:", son.shape)        # bitta son: o'lchami bo'sh

**Ikkinchi: ro’yxat** (bir qator sonlar). `shape` nechta son borligini aytadi.

In [ ]:
royxat = torch.tensor([1.0, 2.0, 3.0])

print(royxat)
print("o'lcham:", royxat.shape)     # 3 ta son

**Uchinchi: jadval** (qatorlar va ustunlar). Rasm shunday saqlanadi: har katak — bitta piksel yorqinligi.

In [ ]:
jadval = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])

print(jadval)
print("o'lcham:", jadval.shape)     # 2 qator, 3 ustun

Natija: `[2, 3]` — **2 qator, 3 ustun**. Bugungi rasmlar **28 × 28** jadval bo’ladi (784 ta piksel).

**Tensorning katta qulayligi:** amal **hamma sonlarga birdan** qo’llanadi. `for` yozish shart emas.

In [ ]:
print(royxat * 2)            # har sonni 2 ga ko'paytirdi
print(royxat + 10)           # har songa 10 qo'shdi
print(royxat * royxat)       # juft-juft ko'paytirdi: 1*1, 2*2, 3*3

Natija: `[2, 4, 6]`, `[11, 12, 13]`, `[1, 4, 9]`. Oxirgisiga qarang: ikki tensor **juft-juft** ko’paytirildi. 6-darsda neyron aynan shunday hisoblagan edi: har javob × o’z og’irligi.

**Sinab ko’ring.** 6-darsdagi futbol qarori: javoblar `x = [1, 0, 1]` (ob-havo yaxshi, do’stlar **yo’q**, uy ishi **bor**), og’irliklar `w = [2, 3, -4]`. `x * w` nima beradi? Avval taxmin qiling.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

`x * w` = `[2, 0, −4]`, `.sum()` hammasini qo’shadi = **−2**. Oxirida `b = −2` ni qo’shsak, **−4**: 6-darsdagi `qaror(1, 0, 1)` javobi (qolamiz). 6-darsda buning uchun `for` tsikli yozgan edik, bugun **bir qator**.

**Mustaqil ish 1.** `neyron(x, w, b)` funksiyasini PyTorch bilan yozing: javoblarni og’irliklarga ko’paytiring, yig’ing va `b` ni qo’shing.

In [ ]:
# TODO: neyron funksiyasini to'ldiring
def neyron(x, w, b):
    jami = 0                      # TODO: (x * w).sum() yozing
    return jami + b

x = torch.tensor([1.0, 1.0, 0.0])
w = torch.tensor([2.0, 3.0, -4.0])
natija1 = neyron(x, w, -2.0)
assert float(natija1) == 3.0, f"neyron natijasi {float(natija1)}, lekin 3 chiqishi kerak. jami = (x * w).sum() yozdingizmi?"
print(natija1)                    # 3 chiqishi kerak

## 2. Autograd — 6-darsdagi `Value` ning katta akasi

6-darsda **40 qator** yozib, `Value` klassini yasadik: u ta’sirlarni (grad) orqaga hisoblardi. PyTorch’da shu **tayyor**, nomi **autograd**.

Faqat bitta narsa kerak: songa **«bu sonning ta’sirini kuzat»** deb belgi qo’yish. Belgi: `requires_grad=True`.

**Avval eng kichik misol:** `L = a × 3`. `a` ni 1 ga oshirsak, `L` 3 ga oshadi. Ta’sir **3** bo’lishi kerak.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)     # a ning ta'sirini kuzat

L = a * 3
print("L =", L.item())

L.backward()                                  # orqaga o'tish (6-darsdagidek)
print("a ning ta'siri (grad):", a.grad.item())

`a.grad = 3.0`. Aynan kutganimiz. Qadamlar 6-darsdagidek: **oldinga** (`L` ni hisobladik), **orqaga** (`L.backward()`), ta’sir `a.grad` ga yozildi.

**Ikki son:** `L = a × b`. 6-darsdagi qoida: har birining ta’siri **boshqasining qiymati**.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

L = a * b
L.backward()

print("a ning ta'siri:", a.grad.item(), " (b ga teng)")
print("b ning ta'siri:", b.grad.item(), " (a ga teng)")

`a.grad = 3` (b ga teng), `b.grad = 2` (a ga teng). 6-darsdagi natija bilan bir xil. Endi 6-darsdagi **uzun hisoblash grafi**: `L = (a · b + c) · f`, bunda a = 2, b = −3, c = 10, f = −2. Qo’lda topgan edik: ta’sirlar **6, −4, −2, 4**.

**Mustaqil ish 2.** `L` formulasini yozing. PyTorch qolganini o’zi qiladi.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(10.0, requires_grad=True)
f = torch.tensor(-2.0, requires_grad=True)

L = a                         # TODO: L = (a * b + c) * f ni yozing
L.backward()

hammasi = [t.grad.item() if t.grad is not None else None for t in (a, b, c, f)]
print("L =", L.item())
print("a, b, c, f ning ta'siri:", hammasi)
assert hammasi == [6.0, -4.0, -2.0, 4.0], f"ta'sirlar {hammasi}, lekin [6, -4, -2, 4] chiqishi kerak. L = (a * b + c) * f yozdingizmi?"

**6, −4, −2, 4**: 6-darsda qo’lda va `Value` bilan topgan javoblarning aynan o’zi. Farq: bu yerda **bitta qator** yozdingiz, grafni PyTorch o’zi yasadi.

**Bir tuzoq:** `.backward()` ni ikki marta chaqirsak, ta’sirlar **qo’shilib** ketadi. Ko’ramiz:

In [ ]:
a = torch.tensor(2.0, requires_grad=True)

L = a * 3
L.backward()
print("1-marta:", a.grad.item())

L = a * 3
L.backward()                       # grad ni nolga tushirmadik
print("2-marta:", a.grad.item())   # 3 + 3 = 6!

Ikkinchi marta **3 emas, 6** chiqdi: eski ta’sir o’chmadi, ustiga qo’shildi. 6-darsda shuning uchun `p.grad = 0.0` yozgan edik (o’qitish tsiklining **3-qadami**). PyTorch’da bu qatorning nomi `optimizer.zero_grad()` bo’ladi.

Endi 6-darsning uy ishini tekshiramiz: `L = (x·w + b − y)²`, x = 2, w = 3, b = 1, y = 5. Qo’lda topgan edingiz: **L = 4, w ning ta’siri 8, b ning ta’siri 4**.

In [ ]:
x, y = 2.0, 5.0
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

L = (x * w + b - y) ** 2
L.backward()

print("L =", L.item())
print("w ning ta'siri:", w.grad.item(), "  b ning ta'siri:", b.grad.item())

**L = 4, w: 8, b: 4**. Uy ishingiz to’g’ri chiqqan bo’lsa, qo’lda hisoblagan edingiz, PyTorch esa bir zumda topdi.

Endi **bitta qadam**. 5- va 6-darsdagi qoida: og’irlikni ta’sirga **teskari** surish. `lr = 0.1`:

In [ ]:
lr = 0.1
with torch.no_grad():                 # bu yerda ta'sir kuzatilmaydi: faqat sonni suramiz
    w -= lr * w.grad                  # w = w - lr * grad
    b -= lr * b.grad

print("yangi w:", round(w.item(), 2), " yangi b:", round(b.item(), 2))
print("yangi L:", round(((x * w + b - y) ** 2).item(), 4))

`w` 3 dan **2.2** ga, `b` 1 dan **0.6** ga kamaydi. Yangi **L = 0**: bir qadamda 4 dan 0 ga tushdi (bitta misol bo’lgani uchun shunday oson chiqdi, katta ma’lumotda bunday bo’lmaydi). Lekin millionlab og’irlikni shunday qo’lda yangilash mumkin emas. Buni PyTorch’ning **optimizer** i qiladi (3-bo’limda).

## 3. `nn.Linear` va `nn.Sequential` — tarmoq 6 qatorda

6-darsda `Neuron` va `MLP` klasslarini o’zimiz yozdik (30 qator). PyTorch’da tayyor:

- **`nn.Linear(3, 1)`** — 3 kirishli **bitta neyron** (`w₁x₁ + w₂x₂ + w₃x₃ + b`);
- **`nn.Linear(2, 4)`** — 2 kirishli **4 ta neyron yonma-yon** (bitta qatlam);
- **`nn.Tanh()`, `nn.ReLU()`** — aktivatsiya;
- **`nn.Sequential(...)`** — qatlamlarni zanjirga ulaydi.

**Avval bitta neyron.** 6-darsdagi futbol qaroridagi og’irliklarni (2, 3, −4 va b = −2) o’zimiz qo’yamiz va javob **3** chiqishini tekshiramiz.

In [ ]:
neyron1 = nn.Linear(3, 1)                                  # 3 kirish -> 1 neyron

with torch.no_grad():
    neyron1.weight.copy_(torch.tensor([[2.0, 3.0, -4.0]]))   # og'irliklar
    neyron1.bias.fill_(-2.0)                                 # b

print("og'irliklar:", neyron1.weight.data)
print("b:", neyron1.bias.data)
print("javob:", neyron1(torch.tensor([1.0, 1.0, 0.0])))      # 3 bo'lishi kerak

Javob **3** (6-darsdagi bilan bir xil). `nn.Linear` ichida aynan `w·x + b` bor. Odatda og’irliklarni **o’zimiz qo’ymaymiz**: ular boshida **tasodifiy**, o’qitish ularni topadi.

Endi 6-darsdagi **lampochka (XOR)** tarmog’ini yasaymiz: 2 kirish → 4 neyron → 1 chiqish.

In [ ]:
torch.manual_seed(1)                       # har safar bir xil tasodifiy sonlar

model = nn.Sequential(
    nn.Linear(2, 4),                       # 2 kirish -> 4 neyron
    nn.Tanh(),                             # aktivatsiya
    nn.Linear(4, 1),                       # 4 neyron -> 1 chiqish
)

print(model)
print("og'irliklar soni:", sum(p.numel() for p in model.parameters()))

**17 ta son**: 6-darsdagi tarmoq bilan **bir xil** (2·4 + 4 + 4·1 + 1). Ya’ni 30 qatorlik klasslarimiz o’rniga **3 qator**.

**6-dars koddan PyTorch’ga tarjima:**

| 6-dars (o’zimiz yozgan) | PyTorch | Vazifasi |
|---|---|---|
| `MLP([2, 4, 1])` | `nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))` | tarmoqni yig’adi |
| `model.parameters()` | `model.parameters()` | hamma og’irlik ro’yxati |
| loss: `(j - y) ** 2` o’rtachasi | `nn.MSELoss()` | xato kvadratining o’rtachasi |
| `p.grad = 0.0` | `optimizer.zero_grad()` | eski ta’sirlarni o’chiradi |
| `loss.backward()` | `loss.backward()` | orqaga o’tish: ta’sirlar |
| `p.data = p.data - lr * p.grad` | `optimizer.step()` | hamma og’irlikni suradi |

Optimizer (haydovchi) yaratilganda unga **qaysi og’irliklarni surish** va **qadam kattaligi** (`lr`) beriladi:

In [ ]:
loss_fn = nn.MSELoss()                                        # loss hisoblagich
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)       # SGD: p.data - lr * p.grad ning o'zi

# lampochka jadvali: 4 holat va to'g'ri javoblar
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

**Avval bitta qadamni ko’ramiz** (6-darsdagi kabi): loss qadamdan oldin va keyin qanday?

In [ ]:
javob = model(X)                     # 1. oldinga
loss = loss_fn(javob, Y)             # 2. loss
print("qadamdan oldin loss:", round(loss.item(), 4))

optimizer.zero_grad()                # 3. eski ta'sirlarni o'chir
loss.backward()                      # 4. orqaga
optimizer.step()                     # 5. qadam

print("qadamdan keyin loss:", round(loss_fn(model(X), Y).item(), 4))

Loss **kamaydi**. 300 qadam shuni takrorlaymiz. Qadamlarning har bir qatori nimaligini bilasiz: 6-darsdagi 5 qadamning aynan o’zi.

**Mustaqil ish 3.** O’qitish tsiklini to’ldiring: 3-, 4- va 5-qadamlar (`zero_grad`, `backward`, `step`).

In [ ]:
torch.manual_seed(1)
model = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
tarix = []

for qadam in range(300):
    javob = model(X)                       # 1. oldinga
    loss = loss_fn(javob, Y)               # 2. loss

    # TODO: 3-qadam: eski ta'sirlarni o'chiring  (optimizer.zero_grad())
    # TODO: 4-qadam: orqaga o'tish               (loss.backward())
    # TODO: 5-qadam: og'irliklarni suring        (optimizer.step())

    tarix.append(loss.item())

for qadam in [0, 50, 100, 200, 299]:
    print(f"{qadam}-qadam: loss = {tarix[qadam]:.4f}")

assert tarix[-1] < 0.01, f"oxirgi loss = {tarix[-1]:.3f}, kamayishi kerak edi. zero_grad, backward va step ni yozdingizmi?"

Loss deyarli **0** ga tushdi. Javoblarni ko’ramiz:

In [ ]:
with torch.no_grad():                    # baholashda ta'sir kuzatilmaydi
    javoblar = model(X)

print("x1 x2 | tarmoq javobi | kerak edi")
for x, j, y in zip(X, javoblar, Y):
    print(f" {int(x[0])}  {int(x[1])} |     {j.item():5.2f}     |   {int(y.item())}")

**0.00, 1.00, 1.00, 0.00**. 6-darsdagi lampochka qoidasi. 6-darsda shu natijaga 85 katak bilan keldik, bugun **6 qator**.

Bu PyTorch’ning asosiy g’oyasi: **tsikl o’sha, faqat qisqa**. Endi uni **haqiqiy** vazifaga qo’llaymiz. Avval tezlik haqida.

## 4. GPU — nega kattaroq vazifalar tez bo’ladi?

Tasavvur qiling: **1000 ta ko’paytirish** kerak.
- **CPU** (telefon/kompyuterning asosiy chipi) — bitta **juda aqlli usta**: murakkab ishni yaxshi qiladi, lekin ishlarni asosan **ketma-ket**.
- **GPU** (grafik chip, o’yin kartasi) — **minglab oddiy ishchi**: har biri faqat ko’paytiradi, lekin **hammasi bir vaqtda**.

Neyron tarmoq — millionlab oddiy ko’paytirish va qo’shish. Aynan GPU’ga mos.

**Colab’da GPU yoqish:** menyuda **Runtime → Change runtime type → T4 GPU → Save**. Keyin kataklarni yuqoridan qayta ishga tushiring. GPU’siz ham dars ishlayveradi, faqat sekinroq.

Avval GPU bormi, tekshiramiz:

In [ ]:
qurilma = "cuda" if torch.cuda.is_available() else "cpu"      # "cuda" = GPU
print("Hozirgi qurilma:", qurilma)

Natija **cuda** bo’lsa, GPU yoqilgan. **cpu** bo’lsa, GPU yo’q (hammasi baribir ishlaydi).

**Tezlik tajribasi:** ikkita katta jadval (2000 × 2000 = 4 million son) ko’paytiriladi. Avval CPU’da:

In [ ]:
A = torch.randn(2000, 2000)         # tasodifiy sonlardan katta jadval
B = torch.randn(2000, 2000)

t0 = time.time()
for _ in range(3):
    C = A @ B                       # jadvallarni ko'paytirish
cpu_vaqt = time.time() - t0
print(f"CPU: {cpu_vaqt:.2f} soniya")

Endi shu hisobni GPU’da (GPU bo’lsa). Faqat **bitta qo’shimcha qadam**: jadvallarni GPU xotirasiga ko’chirish, `.to(qurilma)`.

In [ ]:
if qurilma == "cuda":
    A_gpu, B_gpu = A.to(qurilma), B.to(qurilma)       # ma'lumot GPU ga ko'chdi
    _ = A_gpu @ B_gpu                                 # "qizdirish": birinchi chaqiruv sekinroq
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(3):
        C = A_gpu @ B_gpu
    torch.cuda.synchronize()                          # GPU tugatguncha kut
    gpu_vaqt = time.time() - t0
    print(f"GPU: {gpu_vaqt:.3f} soniya  ->  CPU dan {cpu_vaqt / gpu_vaqt:.0f} marta tez")
else:
    print("GPU yo'q. Menyu: Runtime -> Change runtime type -> T4 GPU, keyin qayta ishga tushiring.")

GPU bo’lsa, **o’nlab marta tezroq**. Esda tuting:
- `.to(qurilma)` — ma’lumot yoki modelni shu qurilmaga ko’chiradi;
- **model va ma’lumot bir qurilmada** bo’lishi shart (ikkalasi ham CPU’da yoki ikkalasi ham GPU’da).

Bugungi tarmoq kichik, shuning uchun CPU’da ham tez ishlaydi. Lekin 8-darsdagi (rasmlar uchun CNN) va undan keyingi darslarda GPU **shart** bo’ladi.

## 5. MNIST: qo’lyozma raqamlar

**MNIST** — mashhur ma’lumot to’plami: odamlar qo’lda yozgan **70 000 ta raqam rasmi** (0 dan 9 gacha). Hamma neyron tarmoq o’quvchisi shundan boshlaydi.

Ma’lumot ikkiga bo’lingan:
- **train (60 000)** — tarmoq shundan **o’rganadi**;
- **test (10 000)** — tarmoq **hech qachon ko’rmagan** rasmlar, **imtihon**.

**Nega ikkiga?** O’qituvchi imtihonda mashq qilgan masalalarini bermaydi: yangi masalalar beradi. Aks holda o’quvchi **yodlab olgan** bo’lishi mumkin, tushunmagan bo’lsa ham. Tarmoq ham shunday: haqiqiy aniqlik **ko’rmagan rasmlarda** o’lchanadi.

Yuklaymiz (birinchi marta bir necha soniya):

In [ ]:
aylantir = transforms.ToTensor()          # rasmni tensorga aylantiradi (piksel 0..1 oralig'ida)

train_data = datasets.MNIST("data", train=True, download=True, transform=aylantir)
test_data = datasets.MNIST("data", train=False, download=True, transform=aylantir)

print("train rasmlar soni:", len(train_data))
print("test rasmlar soni: ", len(test_data))

**60 000** va **10 000**. Bitta rasmni ochib ko’ramiz: u nima?

In [ ]:
rasm, javob = train_data[0]                  # birinchi rasm va uning javobi

print("javob (haqiqiy raqam):", javob)
print("rasm o'lchami:", rasm.shape)          # [1, 28, 28]: 1 kanal (oq-qora), 28 qator, 28 ustun
print("eng qorong'i piksel:", rasm.min().item(), " eng yorqin:", rasm.max().item())

- **o’lcham `[1, 28, 28]`**: 1 kanal (oq-qora rasm), 28 qator, 28 ustun;
- **piksel 0 dan 1 gacha**: 0 — qora, 1 — oq, oralig’i — kulrang.

Rasmning o’zini chizamiz:

In [ ]:
plt.imshow(rasm[0], cmap="gray")          # rasm[0]: 28 x 28 jadval
plt.title(f"Bu raqam: {javob}")
plt.axis("off")
plt.show()

Kompyuter uchun bu rasm — **28 × 28 = 784 ta son** (har biri 0…1). Tarmoq rasmni ko’rmaydi, **784 ta sonni** oladi. Tarmoq kirishi **784 ta** bo’ladi.

Endi 10 ta tasodifiy rasmni to’g’ri javoblari bilan ko’ramiz, odam yozuvi qanchalik xilma-xil ekanini ko’rish uchun:

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(10):
    rasm, javob = train_data[i * 7]                     # har 7-rasmdan
    plt.subplot(1, 10, i + 1)
    plt.imshow(rasm[0], cmap="gray")
    plt.title(javob)
    plt.axis("off")
plt.show()

Yozuvlar har xil: qiyshiq, yo’g’on, ingichka. Tarmoq shu **xilma-xillikdan** o’rganadi.

**Sinab ko’ring.** O’zingiz tanlang: `train_data[100]` qaysi raqam? Avval rasmni ko’rib o’zingiz taxmin qiling, keyin javobni tekshiring.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 6. DataLoader va batch: ma’lumotni bo’lib berish

60 000 rasmni **bir yo’la** tarmoqqa berish — xotira uchun og’ir va qadamlar kam bo’ladi. Bittalab berish esa juda sekin. Yechim: **kichik to’plamlar** (inglizcha *batch*).

Misol: 60 000 ta mashq savoli. Hammasini bir kunda yechmaysiz. 64 tadan **daftarcha** qilib, har daftarchadan keyin dam olib, xatolaringizni tuzatasiz. Tarmoq ham: **64 ta rasm** ko’radi → og’irliklarni biroz suradi → keyingi 64 ta.

- **batch** — bitta daftarcha (bizda 64 ta rasm);
- **epoxa** — **hamma** daftarchalar bir marta tugashi (60 000 rasmning hammasi ko’rildi).

`DataLoader` — shu daftarchalarni tayyorlab beradigan yordamchi.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)     # shuffle: har epoxada aralashtiradi
test_loader = DataLoader(test_data, batch_size=1000)

print("train da batch lar soni:", len(train_loader))      # 60000 / 64 = 938

**938 ta daftarcha** (60 000 ÷ 64, yuqoriga yaxlitlanadi). `shuffle=True` — har epoxada rasmlar **aralashtiriladi**, shunda tarmoq tartibni yodlab olmaydi.

Bitta batch ichiga qaraymiz:

In [ ]:
x, y = next(iter(train_loader))          # birinchi batch

print("rasmlar:", x.shape)               # [64, 1, 28, 28]: 64 ta rasm
print("javoblar:", y.shape)              # [64]: 64 ta raqam
print("birinchi 10 ta javob:", y[:10].tolist())

Bitta batch: **64 ta rasm** va **64 ta javob**. O’qitish tsikli shunday ishlaydi: har batch uchun 5 qadam (oldinga → loss → nolga → orqaga → qadam), 938 marta = **1 epoxa**.

## 7. Modelni yasaymiz

Tarmoq tuzilishi:

| Qatlam | Nima qiladi |
|---|---|
| `nn.Flatten()` | 28 × 28 jadvalni **784 ta sonli qatorga** yoyadi |
| `nn.Linear(784, 128)` | 784 kirish → **128 neyron** (yashirin qatlam) |
| `nn.ReLU()` | aktivatsiya (6-darsdagi turniket) |
| `nn.Linear(128, 10)` | 128 → **10 chiqish**: har raqam uchun bittadan «ball» |

**10 ta chiqish**: nolga, birga, … to’qqizga ball. **Eng baland ball** qaysi raqamda bo’lsa, tarmoq shuni javob qiladi.

**Mustaqil ish 4.** Avval kichikroq vazifa. Modelni yasashdan oldin **aniqlik** hisoblashni o’rganamiz: tarmoq javobi (`bashorat`) va haqiqiy javob (`javob`) ro’yxatlaridan nechtasi to’g’ri kelganini sanang.

In [ ]:
# TODO: to'g'ri javoblar sonini hisoblang
def togri_soni(bashorat, javob):
    return 0                       # TODO: (bashorat == javob).sum().item() yozing

# tekshiruv: 4 tadan 3 tasi to'g'ri
natija = togri_soni(torch.tensor([1, 2, 3, 4]), torch.tensor([1, 2, 0, 4]))
assert natija == 3, f"togri_soni = {natija}, lekin 3 chiqishi kerak. (bashorat == javob).sum().item() yozdingizmi?"
print("to'g'ri:", natija, "ta, 4 tadan")

`bashorat == javob` — juft-juft solishtiradi (`True` / `False`), `.sum()` to’g’rilarini sanaydi. Endi shundan butun test to’plami uchun **aniqlik** funksiyasini yasaymiz. Tayyor, faqat o’qing:

In [ ]:
def aniqlik(model, loader):
    model.eval()                                       # baholash rejimi
    togri, jami = 0, 0
    with torch.no_grad():                              # baholashda ta'sir kerak emas: tezroq
        for x, y in loader:
            x, y = x.to(qurilma), y.to(qurilma)        # ma'lumotni modelning qurilmasiga
            ballar = model(x)                          # har rasm uchun 10 ta ball
            bashorat = ballar.argmax(dim=1)            # eng baland ballli raqam
            togri += togri_soni(bashorat, y)
            jami += len(y)
    return togri / jami

**Kod → vazifasi:**

| Kod | Vazifasi |
|---|---|
| `model.eval()` | baholash rejimi (o’rganmaymiz, faqat tekshiramiz) |
| `torch.no_grad()` | ta’sirlar hisoblanmaydi: tezroq, xotira kam |
| `.to(qurilma)` | rasm va javobni modelning qurilmasiga ko’chirish |
| `model(x)` | har rasm uchun 10 ta ball |
| `.argmax(dim=1)` | 10 ball ichidan **eng baland ball turgan joy** = tarmoq javobi |
| `togri / jami` | to’g’ri javoblar ulushi |

**Mustaqil ish 5.** Modelni o’zingiz yasang. `yashirin` — yashirin qatlamdagi neyronlar soni (bizda 128).

In [ ]:
# TODO: model_yasa funksiyasini to'ldiring
def model_yasa(yashirin=128):
    return nn.Sequential(
        nn.Flatten(),
        # TODO: nn.Linear(784, yashirin), nn.ReLU(), nn.Linear(yashirin, 10) ni qo'shing
    )

model = model_yasa()
soni = sum(p.numel() for p in model.parameters())
assert soni == 101770, f"modelda {soni} ta son bor, lekin 101770 bo'lishi kerak. Flatten dan keyin Linear(784, yashirin), ReLU, Linear(yashirin, 10) ni qo'shdingizmi?"
print("og'irliklar soni:", soni)

**101 770 ta son**: 784·128 + 128 + 128·10 + 10. 6-darsdagi tarmoqda 17 ta edi, bu esa **6000 marta kattaroq**, lekin tuzilishi **bir xil**.

Hali o’qitmagan modelning aniqligi qancha? Tarmoq tasodifiy sonlar bilan boshlaydi, demak **taxminan tasodifiy javob** beradi. 10 ta raqamdan tasodifiy topish ehtimoli **10%**.

In [ ]:
torch.manual_seed(1)
model = model_yasa().to(qurilma)

print(f"o'qitilmagan model aniqligi: {aniqlik(model, test_loader) * 100:.1f}%")     # ~10%

**Taxminan 10%** (8–13%): tasodifiy topish bilan teng. Demak, hozir tarmoq hech narsani bilmaydi. O’qitishdan keyin ~97% bo’lishi kerak. Birinchi qadam: loss qanday hisoblanadi?

## 8. Loss: tarmoq qanchalik adashdi?

6-darsda javob **bitta son** edi (0 yoki 1), loss = xato kvadrati. Bu yerda tarmoq **10 ta ball** beradi. Qaror: **to’g’ri raqamning balli boshqalardan baland bo’lsin**.

Buning uchun avval ballarni **foizga** aylantiramiz (hammasi 100% ga teng): bu **softmax**. Misol: 3 ta ball.

In [ ]:
ballar = torch.tensor([2.0, 1.0, 0.1])

foizlar = torch.softmax(ballar, dim=0)
print("ballar:", ballar.tolist())
print("foizlar:", [round(p, 2) for p in foizlar.tolist()], " yig'indi:", round(foizlar.sum().item(), 2))

Ball **2.0** — **66%** ehtimol, ball 1.0 — 24%, 0.1 — 10%. Yig’indi 100%. **Baland ball = katta ishonch.**

Endi loss: to’g’ri raqamga tarmoq **qanchalik ishongan**? Agar to’g’ri raqam uchun ishonch **kichik** bo’lsa, loss **katta** bo’lsin. Formula: **−ln(p)**, bunda p — to’g’ri raqamga ishonch.

In [ ]:
import math

for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"to'g'ri raqamga ishonch {p:>4} -> loss = {-math.log(p):.2f}")

- ishonch **0.99** (deyarli aniq) → loss **0.01**: yaxshi;
- ishonch **0.1** (tasodif) → loss **2.30**: yomon;
- ishonch **0.01** (qat’iy xato) → loss **4.61**: juda yomon.

Bu **cross-entropy** loss. PyTorch’da bitta qator: `nn.CrossEntropyLoss()` (u softmax va −ln ni o’zi qiladi). Hozirgi o’qitilmagan model uchun loss **~2.3** bo’lishi kerak (tasodif):

In [ ]:
loss_fn = nn.CrossEntropyLoss()

x, y = next(iter(train_loader))
with torch.no_grad():
    loss = loss_fn(model(x.to(qurilma)), y.to(qurilma))

print("o'qitilmagan model loss:", round(loss.item(), 2))     # ~2.3

Loss **~2.3** (aynan −ln(0.1) ga yaqin): tarmoq hamma raqamga teng ishonadi. O’qitish shu sonni **0 ga yaqin** qilishi kerak.

## 9. O’qitamiz

Hamma qism tayyor. O’qitish tsikli **3-bo’limdagi bilan bir xil**, faqat tashqarida **batch lar** va **epoxalar** tsikli bor. Ham optimizer endi **Adam**: u SGD ning aqlliroq turi (qadam kattaligini o’zi moslaydi). Siz uchun farq faqat bitta qator.

| Kod | Vazifasi |
|---|---|
| `for epoxa in range(epochlar)` | butun ma’lumotni `epochlar` marta ko’rib chiqamiz |
| `for x, y in train_loader` | 64 tadan batch olamiz |
| `x, y = x.to(qurilma), y.to(qurilma)` | ma’lumotni modelning qurilmasiga ko’chiramiz |
| `ballar = model(x)` | **1. oldinga** |
| `loss = loss_fn(ballar, y)` | **2. loss** |
| `optimizer.zero_grad()` | **3.** eski ta’sirlarni o’chir |
| `loss.backward()` | **4. orqaga** |
| `optimizer.step()` | **5. qadam** |
| `aniqlik(model, test_loader)` | har epoxadan keyin **imtihon** |

In [ ]:
def o_qit(model, epochlar=3, lr=0.001):
    model = model.to(qurilma)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    natijalar = []

    for epoxa in range(epochlar):
        model.train()                                      # o'rganish rejimi
        jami_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(qurilma), y.to(qurilma)
            ballar = model(x)                              # 1. oldinga
            loss = loss_fn(ballar, y)                      # 2. loss
            optimizer.zero_grad()                          # 3. eski ta'sirlarni o'chir
            loss.backward()                                # 4. orqaga
            optimizer.step()                               # 5. qadam
            jami_loss += loss.item()

        oraliq_loss = jami_loss / len(train_loader)        # o'rtacha loss
        aniq = aniqlik(model, test_loader)                 # imtihon
        natijalar.append((oraliq_loss, aniq))
        print(f"{epoxa + 1}-epoxa: loss = {oraliq_loss:.4f}, test aniqligi = {aniq * 100:.2f}%")
    return natijalar

**3 epoxa** (har epoxa taxminan 10–30 soniya; qanday o’sishini kuzating):

In [ ]:
torch.manual_seed(1)
model = model_yasa()
natijalar = o_qit(model, epochlar=3)

assert natijalar[-1][1] > 0.95, f"aniqlik {natijalar[-1][1] * 100:.1f}%, lekin 95% dan yuqori bo'lishi kerak edi. Oldingi mustaqil ishlarni (4, 5) tekshiring."

Birinchi epoxadayoq aniqlik **~10% dan ~94% ga** chiqdi, uchinchisida **96–97%**: **95% dan yuqori**. Tarmoq 60 000 rasmdan raqamni tanishni **o’zi o’rgandi**. Hech kim «3 ning shakli bunday» deb yozmadi.

**Muhim:** bu aniqlik **test** to’plamida, tarmoq **hech ko’rmagan** rasmlarda. Demak, u yodlamadi, **o’rgandi**.

Loss va aniqlikning o’zgarishini grafikda ko’ramiz:

In [ ]:
losslar = [n[0] for n in natijalar]
aniqliklar = [n[1] * 100 for n in natijalar]

plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(losslar) + 1), losslar, marker="o")
plt.xlabel("epoxa"); plt.ylabel("loss"); plt.title("Loss kamayadi")

plt.subplot(1, 2, 2)
plt.plot(range(1, len(aniqliklar) + 1), aniqliklar, marker="o", color="green")
plt.xlabel("epoxa"); plt.ylabel("aniqlik, %"); plt.title("Aniqlik oshadi")
plt.tight_layout()
plt.show()

Loss tushadi, aniqlik oshadi: avval tez, keyin sekin. 5-darsdagi loss grafigi bilan bir xil shakl.

**Sinab ko’ring.** Yashirin qatlamda **faqat 2 ta** neyron bo’lsa, aniqlik nechaga yetadi? Avval taxmin qiling: 128 ta bilan 97%. 2 ta bilan-chi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

2 ta neyron bilan aniqlik **ancha past** (taxminan 40–50%). Neyronlar kam bo’lsa, tarmoq 10 ta raqamni ajratishga **yetmaydi**: xuddi 6-darsda bitta neyron lampochka qoidasiga yetmagani kabi.

**Sinab ko’ring.** Qadam kattaligi `lr = 1.0` (juda katta) bo’lsa nima bo’ladi? 5-darsni eslang: katta qadam…

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

`lr = 1.0` bilan loss **portladi** (o’nlab marta kattalashdi), aniqlik **10–15%** da qoldi. 5-darsdagi **«qadam juda katta»** hodisasi: tarmoq vodiy tubidan o’tib ketadi va yaqinlasha olmaydi. Qadam kattaligi **muhim**.

## 10. Tarmoq qanday javob beradi?

Test to’plamidan bitta rasmni olamiz va tarmoq **10 ta raqamga qancha ishonishini** ko’ramiz. Ishlatadigan modelimiz: 9-bo’limda 3 epoxa o’qitilgan `model`.

In [ ]:
def tahlil(i):
    rasm, javob = test_data[i]
    model.eval()
    with torch.no_grad():
        ballar = model(rasm.unsqueeze(0).to(qurilma))        # bitta rasm: batch ko'rinishiga keltiramiz
        foizlar = torch.softmax(ballar, dim=1)[0].cpu()      # 10 ta ishonch foizi

    taxmin = foizlar.argmax().item()
    plt.figure(figsize=(8, 2.8))
    plt.subplot(1, 2, 1)
    plt.imshow(rasm[0], cmap="gray"); plt.axis("off")
    plt.title(f"haqiqiy: {javob}  |  tarmoq: {taxmin}")
    plt.subplot(1, 2, 2)
    plt.bar(range(10), foizlar * 100)
    plt.xticks(range(10)); plt.xlabel("raqam"); plt.ylabel("ishonch, %")
    plt.tight_layout()
    plt.show()

tahlil(0)

Chapda rasm va javoblar, o’ngda **10 ta ustun**: tarmoq har raqamga qancha ishonishi. Odatda bitta ustun **juda baland** (95–100%).

**Sinab ko’ring.** Boshqa rasmni tanlang: `tahlil(5)`, `tahlil(100)`, `tahlil(1234)`. Avval rasmga qarab o’zingiz taxmin qiling, keyin tarmoq bilan solishtiring.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 11. Tarmoq qayerda adashadi?

Aniqlik ~97%: **100 tadan 3 tasi xato**. 10 000 ta test rasmidan ~300 tasi. Eng qiziq savol: **qaysilarida**? Hamma test rasmlarini bir yo’la tekshirib, xatolarini yig’amiz:

In [ ]:
model.eval()
X_test = test_data.data.float().unsqueeze(1) / 255            # [10000, 1, 28, 28], piksel 0..1
Y_test = test_data.targets                                    # to'g'ri javoblar

with torch.no_grad():
    bashorat = model(X_test.to(qurilma)).argmax(dim=1).cpu()

xato_indekslar = (bashorat != Y_test).nonzero().flatten()    # xato qilingan rasmlar raqami
print("xato qilingan rasmlar soni:", len(xato_indekslar), "/ 10000")

Taxminan **250–350** ta xato. Ularning 12 tasini ko’ramiz: har rasm tepasida **«haqiqiy → tarmoq»**.

In [ ]:
plt.figure(figsize=(12, 3.2))
for k, i in enumerate(xato_indekslar[:12]):
    plt.subplot(2, 6, k + 1)
    plt.imshow(test_data.data[i], cmap="gray")
    plt.title(f"{Y_test[i].item()} -> {bashorat[i].item()}", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

Ko’p xatolarda **odam ham ikkilanadi**: 4 va 9, 8 va 3, 7 va 2 o’xshash yozilgan, ba’zilarining yozuvi juda xunuk. Bu **normal**: ~97% — yaxshi natija. 100% — deyarli yo’q (odam ham 100% to’g’ri o’qiy olmaydi).

**Sinab ko’ring.** Qaysi juftlikni tarmoq eng ko’p **adashtiradi**? Sanab ko’ramiz (haqiqiy → tarmoq):

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Eng ko’p adashadigan juftliklarda **o’xshash shaklli raqamlar** turadi: **4 → 9** (eng ko’p), 8 → 3, 8 → 5, 7 → 2. Tarmoq inson kabi shakllarni solishtiradi, shuning uchun o’xshashlarda adashadi.

## 12. Modelni saqlash va yuklash

Tarmoqni 3 daqiqa o’qitdingiz. Har safar qayta o’qitish shart emas: **o’rgangan og’irliklarni faylga saqlash** mumkin. Fayl kichik: 101 770 ta son (taxminan 400 KB).

Saqlaymiz:

In [ ]:
torch.save(model.state_dict(), "raqam_modeli.pt")          # state_dict: hamma og'irliklar ro'yxati

import os
print("saqlandi:", os.path.getsize("raqam_modeli.pt") // 1024, "KB")

Endi **yangi** (o’qitilmagan) model yasab, unga saqlangan og’irliklarni yuklaymiz. Aniqlik **bir xil** chiqishi kerak:

In [ ]:
yangi = model_yasa().to(qurilma)
print(f"yuklashdan oldin:  {aniqlik(yangi, test_loader) * 100:.1f}%")      # ~10%

yangi.load_state_dict(torch.load("raqam_modeli.pt", map_location=qurilma))   # saqlangan og'irliklarni yukla
print(f"yuklashdan keyin:  {aniqlik(yangi, test_loader) * 100:.1f}%")      # ~97%

Yuklashdan oldin **~10%**, keyin **~97%**. Tarmoq «bilimi» faylda. Shunday ilovalar ishlaydi: model **bir marta** o’qitiladi, keyin millionlab telefonlarga fayl sifatida yuboriladi.

Colab’dan faylni kompyuteringizga yuklab olish (ixtiyoriy):

In [ ]:
try:
    from google.colab import files
    files.download("raqam_modeli.pt")
except ImportError:
    print("Bu Colab emas: fayl shu papkada turibdi.")

## 13. Siz buni har kuni ishlatasiz

Bugungi tarmoq: **101 770 ta son**, 60 000 rasmdan o’rgandi. Xuddi shu tsikl, faqat kattaroq:
- **Telefon kamerasi matnni o’qishi**: rasmdagi yozuvni harflarga aylantiradi;
- **Bank ilovasi**: chekdagi summani rasmdan o’qiydi;
- **Pochta**: konvertdagi indeksni mashina o’zi o’qiydi (MNIST aslida shunday vazifadan chiqqan);
- **Face ID**: yuzingiz o’rniga raqam, xuddi shu qoida.

Hammasi **bir tsikl**: **oldinga → loss → orqaga → qadam**. 5-darsda `k` va `b` (2 ta son), 6-darsda 17 ta son, bugun 101 770 ta son, ChatGPT’da yuz milliardlab. **Qoida o’zgarmaydi.**

**Keyingi dars (8-dars):** **CNN**. Bugungi tarmoq rasmni 784 ta sonli **qator** deb oldi va qo’shni piksellar qo’shni ekanini bilmadi. CNN rasmni **rasm sifatida** ko’radi: shuning uchun yanada aniqroq.

## Uy ishi

**1. Fashion-MNIST.** Xuddi shu model, lekin raqam o’rniga **kiyim rasmlari** (futbolka, shim, krossovka…). Quyidagi shablonni ishga tushiring: aniqlik nechaga yetdi? (Taxminan 85–88%.) Nega raqamlarga qaraganda **pastroq**?

**2. Aniqlikni oshiring.** Quyidagilardan kamida **ikkitasini** sinab, qaysi biri eng yaxshi natija berganini yozing:
- yashirin neyronlar soni: 64, 128, 256, 512;
- epoxalar soni: 3, 5, 10;
- `lr`: 0.01, 0.001, 0.0001;
- **ikkinchi yashirin qatlam** qo’shish.

**3. Qisqa yozma javob (2–3 gap).** Nega aniqlikni **train** da emas, **test** da o’lchaymiz?

Boshlash uchun shablon (1-vazifa):

In [ ]:
fashion_train = datasets.FashionMNIST("data", train=True, download=True, transform=aylantir)
fashion_test = datasets.FashionMNIST("data", train=False, download=True, transform=aylantir)

train_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)      # o'qitish shu loader dan oladi
test_loader = DataLoader(fashion_test, batch_size=1000)

nomlar = ["futbolka", "shim", "sviter", "ko'ylak", "palto", "sandal", "kofta", "krossovka", "sumka", "botinka"]
rasm, javob = fashion_train[0]
plt.imshow(rasm[0], cmap="gray"); plt.title(nomlar[javob]); plt.axis("off"); plt.show()

torch.manual_seed(1)
fashion_model = model_yasa()
o_qit(fashion_model, epochlar=3)

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy): Adam yoki SGD?

Biz **Adam** ishlatdik. 3-bo’limdagi **SGD** (`p.data - lr * p.grad`) bilan solishtiring:
1. `o_qit` ichidagi `torch.optim.Adam(...)` ni `torch.optim.SGD(model.parameters(), lr=lr)` ga almashtiring (MNIST uchun `lr = 0.1` qo’ying).
2. Ikkalasini 3 epoxa o’qiting va aniqlikni solishtiring.

**Savol:** qaysi biri tezroq o’rgandi? Nega Adam ko’pincha tezroq (maslahat: u har og’irlik uchun qadamni **alohida** moslaydi)?

## Yakun

Bugun siz:
- **tensor** nima ekanini bildingiz: sonlar jadvali, rasm ham shunday (28 × 28);
- PyTorch **autograd** 6-darsdagi `Value` ning aynan o’zi ekanini, ta’sirlarni bir qatorda topishini ko’rdingiz;
- 6-darsdagi tarmoqni **6 qatorda** qayta yasadingiz (`nn.Linear`, `nn.Sequential`, `optimizer`);
- **DataLoader**, **batch** va **epoxa** nima ekanini tushundingiz;
- **60 000 rasmdan raqam tanishni** o’rgangan, **95% dan yuqori** aniqlikdagi tarmoqni o’qitdingiz;
- tarmoq **qayerda adashishini** ko’rdingiz va modelni **saqlab, yukladingiz**.

Keyingi darsda (8-dars) **CNN**: rasmni sonlar qatori emas, **rasm sifatida** ko’radigan tarmoq.